In [ ]:
import os
import requests
import xml.etree.ElementTree as ET
import fitz
from tqdm import tqdm
import uuid
import json

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.schema import Document
from langchain.embeddings import HuggingFaceEmbeddings

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    TrainerCallback
)
from peft import LoraConfig
from trl import SFTTrainer
import torch
import time

from huggingface_hub import login
from google.colab import userdata 

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("Hugging Face token loaded from Colab secrets.")
except Exception as e:
    print(f"Error loading HF_TOKEN from Colab secrets: {e}")
    print("Please ensure you have set HF_TOKEN in Colab secrets with notebook access enabled.")
    

login() # authenticate with HFace
print("Hugging Face login complete.") 

model_name = "google/gemma-2b-it"
data_path = "llm_qa.jsonl"

print("=" * 80)
print("STARTING QLoRA TRAINING")
print("=" * 80)
print("Model:", model_name)
print("Dataset:", data_path)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

print("=" * 80)

print("Loading dataset...")

dataset = load_dataset("json", data_files=data_path)

print(dataset)
print("Training samples:", len(dataset["train"]))

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_name, token = 'hf_xxxxxxxxxxxxxx') 
tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    user_content = next((msg['content'] for msg in example['messages'] if msg['role'] == 'user'), '')
    assistant_content = next((msg['content'] for msg in example['messages'] if msg['role'] == 'assistant'), '')
    return {
        "text": f"<s>[INST] {user_content} [/INST] {assistant_content}</s>"
    }

print("Formatting dataset...")
dataset = dataset.map(format_example, remove_columns=["messages"]) 

print("Dataset features after formatting:", dataset["train"].features)

print("Example formatted row:")
print(dataset["train"][0]["text"][:500])


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    #load_in_4bit=True,
    device_map="auto"
)

print("Model loaded successfully.")

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./llm_finetuned",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    num_train_epochs=2,

    logging_steps=1,                 # print often
    save_strategy="epoch",

    bf16=True if torch.cuda.is_available() else False,
    fp16=False,

    report_to="none",

    # dataloader_num_workers=2,
    dataloader_num_workers=0,
    ddp_find_unused_parameters=False,


    logging_first_step=True
)

class ProgressPrinter(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print("=" * 80)
        print("TRAINING STARTED")
        print("Total steps:", state.max_steps)
        print("=" * 80)

    def on_log(self, args, state, control, logs=None, **kwargs):
        elapsed = time.time() - self.start_time

        step = state.global_step
        total = state.max_steps

        loss = logs.get("loss") if logs else None
        lr = logs.get("learning_rate") if logs else None

        pct = (step / total) * 100 if total > 0 else 0

        # Safe formatting
        loss_str = f"{loss:.4f}" if loss is not None else "N/A"
        lr_str = f"{lr:.8f}" if lr is not None else "N/A"

        print(
            f"[Step {step}/{total}] {pct:.2f}% | Loss: {loss_str} | LR: {lr_str} | Elapsed: {elapsed/60:.2f} min"
            )

    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"GPU Memory Used: {mem:.2f} GB")

    def on_epoch_end(self, args, state, control, **kwargs):
        print("=" * 80)
        print(f"Finished Epoch {state.epoch:.2f}")
        print("=" * 80)

    def on_train_end(self, args, state, control, **kwargs):
        print("=" * 80)
        print("TRAINING COMPLETE")
        print("=" * 80)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    peft_config=peft_config,
    args=training_args,
    #dataset_text_field="text"
)

trainer.add_callback(ProgressPrinter())

trainer.train()

print("Saving final adapter...")
trainer.save_model("./llm_finetuned_final")

print("DONE.")
